In [5]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv, Node2Vec, GCNConv
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
import torch_geometric.transforms as T

dataset = Planetoid(root="data/Planetoid", name="PubMed")

data = dataset[0]

print(data)


Data(x=[19717, 500], edge_index=[2, 88648], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717])


C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  if osp.exists(f) and torch.lo

In [7]:
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=False
)

train_data, val_data, test_data = transform(data)

train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)



In [29]:
# DeepWalk is the same as Node2Vec with p=1 and q=1

dw_model = Node2Vec(
    train_data.edge_index,
    embedding_dim=64,
    walk_length=10,
    context_size=7,
    walks_per_node=10,
    num_negative_samples=1,
    p=1,
    q=1,
    sparse=True
).to(device)

loader = dw_model.loader(batch_size=256, shuffle=True)
dw_optimizer = torch.optim.SparseAdam(dw_model.parameters(), lr=0.01)


In [30]:
for epoch in range(1, 201):

    loss = train_deepwalk()

    if epoch % 5 == 0:

        val_auc, val_ap = evaluate_deepwalk(val_data)

        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Val AUC {val_auc:.4f} | Val AP {val_ap:.4f}")
        dw_auc, dw_ap = evaluate_deepwalk(test_data)

        print("AUC:", dw_auc)
        print("AP :", dw_ap)

Epoch 005 | Loss 1.6448 | Val AUC 0.7034 | Val AP 0.7297
AUC: 0.7237726996148783
AP : 0.7457837699464175
Epoch 010 | Loss 0.9735 | Val AUC 0.7907 | Val AP 0.8413
AUC: 0.8111956172454353
AP : 0.8550755820433917
Epoch 015 | Loss 0.8582 | Val AUC 0.8121 | Val AP 0.8659
AUC: 0.8258172635183568
AP : 0.8739635711678307
Epoch 020 | Loss 0.8234 | Val AUC 0.8173 | Val AP 0.8720
AUC: 0.8298950028265063
AP : 0.8789961050080044
Epoch 025 | Loss 0.8090 | Val AUC 0.8234 | Val AP 0.8771
AUC: 0.8301670387825985
AP : 0.8797674294551251
Epoch 030 | Loss 0.8017 | Val AUC 0.8231 | Val AP 0.8772
AUC: 0.8322237650122509
AP : 0.881681030909617
Epoch 035 | Loss 0.7977 | Val AUC 0.8254 | Val AP 0.8789
AUC: 0.8331981765776303
AP : 0.8822825262178761
Epoch 040 | Loss 0.7957 | Val AUC 0.8252 | Val AP 0.8790
AUC: 0.8333014468942643
AP : 0.8826661379631326
Epoch 045 | Loss 0.7950 | Val AUC 0.8235 | Val AP 0.8788
AUC: 0.8318115492105331
AP : 0.8814973230178704
Epoch 050 | Loss 0.7935 | Val AUC 0.8242 | Val AP 0.8792

KeyboardInterrupt: 

In the above cell after running a few times the training with different configurations and monitoring the outputs I picked the best number of epochs and configuration.

In [31]:
def train_deepwalk():

    dw_model.train()
    total_loss = 0

    for pos_rw, neg_rw in loader:

        dw_optimizer.zero_grad()

        loss = dw_model.loss(pos_rw.to(device), neg_rw.to(device))

        loss.backward()
        dw_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate_deepwalk(data):

    dw_model.eval()

    z = dw_model()
    z = F.normalize(z, p=2, dim=1)

    out = (z[data.edge_label_index[0]] * z[data.edge_label_index[1]]).sum(dim=-1).sigmoid()

    y_true = data.edge_label.cpu().numpy()
    y_pred = out.cpu().numpy()

    auc = roc_auc_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)

    return auc, ap


print("\nTraining DeepWalk")


num_runs = 50
test_aucs = []
test_aps = []

for run in range(1, num_runs + 1):

    dw_model = Node2Vec(
    train_data.edge_index,
    embedding_dim=64,
    walk_length=10,
    context_size=7,
    walks_per_node=10,
    num_negative_samples=1,
    p=1,
    q=1,
    sparse=True
    ).to(device)

    loader = dw_model.loader(batch_size=256, shuffle=True)
    dw_optimizer = torch.optim.SparseAdam(dw_model.parameters(), lr=0.01)
    for epoch in range(1, 41):
        loss = train_deepwalk()
        
    dw_auc, dw_ap = evaluate_deepwalk(test_data)
    test_aucs.append(dw_auc)
    test_aps.append(dw_ap)
    print(f"Run {run:02d} Completed - Test AUC: {dw_auc:.4f}, Test AP: {dw_ap:.4f}")

print(f"\n--- Final Results over {num_runs} runs ---")
print(f"Test AUC: {np.mean(test_aucs):.4f} ± {np.std(test_aucs):.4f}")
print(f"Test AP:  {np.mean(test_aps):.4f} ± {np.std(test_aps):.4f}")


Training DeepWalk
Run 01 Completed - Test AUC: 0.8325, Test AP: 0.8825
Run 02 Completed - Test AUC: 0.8310, Test AP: 0.8808
Run 03 Completed - Test AUC: 0.8341, Test AP: 0.8830
Run 04 Completed - Test AUC: 0.8280, Test AP: 0.8798
Run 05 Completed - Test AUC: 0.8287, Test AP: 0.8812
Run 06 Completed - Test AUC: 0.8325, Test AP: 0.8824
Run 07 Completed - Test AUC: 0.8297, Test AP: 0.8813
Run 08 Completed - Test AUC: 0.8347, Test AP: 0.8833
Run 09 Completed - Test AUC: 0.8308, Test AP: 0.8811
Run 10 Completed - Test AUC: 0.8317, Test AP: 0.8824
Run 11 Completed - Test AUC: 0.8321, Test AP: 0.8826
Run 12 Completed - Test AUC: 0.8305, Test AP: 0.8816
Run 13 Completed - Test AUC: 0.8376, Test AP: 0.8857
Run 14 Completed - Test AUC: 0.8342, Test AP: 0.8841
Run 15 Completed - Test AUC: 0.8344, Test AP: 0.8841
Run 16 Completed - Test AUC: 0.8314, Test AP: 0.8817
Run 17 Completed - Test AUC: 0.8311, Test AP: 0.8812
Run 18 Completed - Test AUC: 0.8309, Test AP: 0.8810
Run 19 Completed - Test AUC